# 4.3

<div class="alert alert-info"> VAE Imputation: Evaluation

## Ziel
Bewertung der Imputationsgüte. 
Es wird verglichen, wie gut die vom VAE rekonstruierten Werte ("Imputed") mit den tatsächlichen Werten ("Original") übereinstimmen.

## Metriken
- **RMSE**: Root Mean Squared Error (Niedriger ist besser)
- **Verteilungs-Check**: Visueller Vergleich der Dichtefunktionen (KDE).

## Output
- Grafiken zur Imputationsqualität pro Feature.
- Zusammenfassung der Fehlerstatistik.
</div>

In [ ]:

# ------------------------- Warnungen unterdrücken -------------------------
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, r2_score
from matplotlib.backends.backend_pdf import PdfPages
import time
import sys
import json
import gc

sns.set_theme(style="whitegrid")


In [ ]:
base_dir = Path.cwd()
inference_root = base_dir.parent / "4.2_Inference" / "Inference_Results"
def log(msg): print(msg)
log(f"Suche Ergebnisse in: {inference_root}")

# ------------------------- Warten auf Ordnererstellung -------------------------
results_folder = None
wait_counter = 0
max_wait_seconds = 1800 

while results_folder is None:
    if not inference_root.exists():
        log("Inferenzordner fehlt noch")
        time.sleep(2)
        continue
        
    timestamp_folders = [f for f in inference_root.iterdir() if f.is_dir()]
    if timestamp_folders:
        candidate = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
        age = time.time() - candidate.stat().st_mtime
        
        if age < 999999:
             results_folder = candidate
             log(f"Nutze Ergebnisse aus: {results_folder.name}")
             break
        else:
             pass
             
    time.sleep(5)
    wait_counter += 5
    if wait_counter >= max_wait_seconds:
        log("Timeout!")
        raise TimeoutError("Keine neuen Ergebnisse von 4.2 gefunden.")

# ------------------------- Setup Loop -------------------------
models_dir_path = results_folder.parent.parent.parent / "4.1_VAE_Imputation" / "Models" / results_folder.name
signal_file_path = models_dir_path / "DONE_TRAINING"
out_dir_eval = base_dir / "Evaluation_Results" / results_folder.name
out_dir_eval.mkdir(parents=True, exist_ok=True)
log(f"Ergebnis gespeichert in: {out_dir_eval}")

processed_runs = {}
summary_stats = []
all_metrics = []
done_training = False

log("\nStarte Monitoring Loop...")

while True:
    
    current_files = list(results_folder.glob("Imputation_Results_*.csv"))
    valid_files = [f for f in current_files if f.stat().st_size > 0]
    
    # ------------------------- Dateien nach Run ID gruppieren -------------------------
    run_groups = {}
    for f in valid_files:
        if "_Level_" not in f.name: continue
        parts = f.stem.split("_Level_")
        base_name = parts[0].replace("Imputation_Results_Run_", "").replace("Imputation_Results_", "")
        if "Trennlinie" in base_name: continue
        if base_name not in run_groups: run_groups[base_name] = []
        run_groups[base_name].append(f)
        
    # ------------------------- Prüfen ob Training fertig ist -------------------------
    if signal_file_path.exists():
        if not done_training:
             log("Signal DONE_TRAINING erkannt.")
             done_training = True

    # ------------------------- Logik  -------------------------
    def get_run_index(run_id):
        parts = run_id.split("_")
        for p in parts:
             if p.isdigit(): return int(p)
        return 99999
    
    all_found_runs = sorted(run_groups.keys(), key=get_run_index)
    
    safe_to_process = []
    if done_training:
        safe_to_process = all_found_runs
    else:
        safe_to_process = all_found_runs
            
    # ------------------------- Neue Runs -------------------------
    new_runs = safe_to_process
    new_runs.sort(key=get_run_index)

    for run_id in new_runs:

        # ------------------------- Aktuellen Run-State abholen -------------------------
        if run_id in run_groups:
            current_files_set = set(f.name for f in run_groups[run_id])
        else:
            continue

        # ------------------------- Prüfen ob Run schon verarbeitet -------------------------
        if run_id in processed_runs and processed_runs[run_id] == current_files_set:
            continue

        # ------------------------- SICHERHEITS-CHECK: Nur verarbeiten wenn nächster Run schon existiert -------------------------
        # (Ausnahme: done_training ist True, dann ist der letzte Run ja auch fertig)
#         curr_idx_in_list = new_runs.index(run_id)
#         is_last_in_list = (curr_idx_in_list == len(new_runs) - 1)
#         
#         if not done_training and is_last_in_list:
#             # Wir sind beim aktuellsten Run, aber Training läuft noch -> Warten bis nächster Run auftaucht
#             # Damit vermeiden wir, dass wir unvollständige Runs immer wieder neu plotten.
#             if wait_counter % 12 == 0: # Nur ca. jede Minute loggen (12 * 5s = 60s)
#                 log(f"Postponing evaluation of {run_id} until next run starts (to ensure completeness)...")
#             continue

        log(f"Evaluation Run: {run_id}")
        
        level_files = run_groups[run_id]
        def get_level(p):
            try: return int(p.stem.split("_Level_")[1])
            except: return 0
        level_files.sort(key=get_level)
        
        pdf_path = out_dir_eval / f"Analysis_Run_{run_id}.pdf"

        rmse_per_feat = {} 
        
        try:
             with PdfPages(pdf_path) as pdf:
                if not level_files: 
                    # ------------------------- Erst als fertig markieren, wenn letzter druch ist -------------------------
                    processed_runs[run_id] = current_files_set
                    continue
                
                # ------------------------- Header-Dateien -------------------------
                header_file = level_files[0]
                df_l1 = pd.read_csv(header_file)
                features = df_l1['Feature'].unique()
                del df_l1
                gc.collect()

                # ------------------------- Seite 1 Übersicht -------------------------
                fig = plt.figure(figsize=(11.69, 8.27))
                gs = fig.add_gridspec(3, 2, height_ratios=[0.25, 0.45, 0.3], wspace=0.3, hspace=0.4)
                
                ax_header = fig.add_subplot(gs[0, :])
                ax_header.axis('off')
                ax_header.text(0.5, 0.7, "VAE Imputation Analysis", fontsize=24, weight='bold', ha='center', color='#2c3e50')
                ax_header.text(0.5, 0.4, f"Run ID: {run_id}", fontsize=14, ha='center', color='#7f8c8d')
                ax_header.text(0.5, 0.2, f"Date: {time.strftime('%Y-%m-%d %H:%M')}", ha='center', fontsize=10)

                # ------------------------- Metadaten Pfad -------------------------
                meta_filename = f"{run_id}_meta.json"

                # ------------------------- Fallback für alte Namen -------------------------
                if not (models_dir_path / meta_filename).exists(): 
                     meta_filename = f"Model_{run_id}_meta.json"
                meta_path = models_dir_path / meta_filename
                vae_info = "Info Unavailable"
                cv_text = "CV: N/A"
                training_loss_history = []
                
                if meta_path.exists():
                    try:
                        with open(meta_path, 'r') as f:
                            meta = json.load(f)
                        vae_info = f"Latent Dim: {meta.get('latent_dim', '?')}\nHidden Dim: {meta.get('hidden_dim', '?')}\nEncoder Layers: {meta.get('enc_layers', '?')}\nDecoder Layers: {meta.get('dec_layers', '?')}\nEpochs: {meta.get('epochs_trained', '?')}\nBatch Size: {meta.get('batch_size', '?')}\nKLD Weight: {meta.get('kld_weight', '?')}"
                        cv_mean = meta.get('cv_mean_score', 'N/A')
                        cv_masked = meta.get('cv_masked_score', 'N/A')
                        if isinstance(cv_masked, float): cv_masked = f"{cv_masked:.4f}"
                        if isinstance(cv_mean, float): cv_mean = f"{cv_mean:.4f}"
                        cv_text = f"CV Mean Loss: {cv_mean}\nMasked CV Loss: {cv_masked}"
                        training_loss_history = meta.get('training_loss_history', [])
                        
                        cv_scores = meta.get('cv_fold_scores', [])
                        if cv_scores:
                            ax_cv = fig.add_subplot(gs[1, 1])
                            sns.barplot(x=[f"Fold {i+1}" for i in range(len(cv_scores))], y=cv_scores, ax=ax_cv, palette="viridis")
                            ax_cv.set_title("Cross-Validation Loss per Fold", fontsize=10)
                            ax_cv.set_ylim(0, max(cv_scores)*1.2)
                            for i, v in enumerate(cv_scores):
                                ax_cv.text(i, v, f"{v:.3f}", ha='center', va='bottom', fontsize=8)
                    except: pass
                
                ax_info = fig.add_subplot(gs[1, 0])
                ax_info.axis('off')
                ax_info.text(0.1, 0.6, vae_info, fontsize=11, linespacing=1.5)
                ax_info.text(0.1, 0.3, "Validation:", weight='bold', fontsize=12)
                ax_info.text(0.1, 0.1, cv_text, fontsize=11)
                
                ax_desc = fig.add_subplot(gs[2, :])
                ax_desc.axis('off')
                ax_desc.text(0, 0.8, "Metrics Explanation:", weight='bold', fontsize=10)
                ax_desc.text(0, 0.5, "- RMSE (Root Mean Squared Error): Measures reconstruction accuracy. Lower is better.", fontsize=10)
                ax_desc.text(0, 0.2, "- R² Score: Measures variance explained. 1.0 is perfect, 0.0 is random.", fontsize=10)

                pdf.savefig(fig)
                plt.close(fig)
                
                # ------------------------- Seite 2 Training Loss History -------------------------
                if training_loss_history:
                    fig_loss = plt.figure(figsize=(11.69, 8.27))
                    ax_loss = fig_loss.add_subplot(111)
                    
                    epochs = range(1, len(training_loss_history) + 1)
                    sns.lineplot(x=epochs, y=training_loss_history, marker='o', color='b', ax=ax_loss)
                    
                    ax_loss.set_title("Training Loss per Epoch", fontsize=16, weight='bold')
                    ax_loss.set_xlabel("Epoch", fontsize=12)
                    ax_loss.set_ylabel("Loss (MSE + KLD)", fontsize=12)
                    ax_loss.grid(True)
                    
                    # ------------------------- Trainig Loss -------------------------
                    if len(training_loss_history) > 0:
                         ax_loss.text(epochs[0], training_loss_history[0], f"{training_loss_history[0]:.4f}", ha='right', va='bottom')
                         ax_loss.text(epochs[-1], training_loss_history[-1], f"{training_loss_history[-1]:.4f}", ha='left', va='top')

                    pdf.savefig(fig_loss)
                    plt.close(fig_loss)

                # ------------------------- Loop der einzelnen "Level" -------------------------
                for lvl_file in level_files:
                    current_level = get_level(lvl_file)
                    log(f"  Processing Level {current_level}...")
                    
                    df = pd.read_csv(lvl_file)
                    for col in df.select_dtypes(include=['float64']).columns: df[col] = df[col].astype('float32')
                    for col in df.select_dtypes(include=['int64']).columns: df[col] = df[col].astype('int32')
                    if df.empty: continue
                    
                    # ------------------------- Trenner zwischen "Leveln" -------------------------
                    fig_sep, ax_sep = plt.subplots(figsize=(11.69, 8.27))
                    ax_sep.text(0.5, 0.5, f"Level {current_level}", ha='center', fontsize=30, weight='bold')
                    pdf.savefig(fig_sep)
                    plt.close(fig_sep)
                    
                    # ------------------------- Plot pro Kombination -------------------------
                    
                    # ------------------------- Nach KOmbination gruppieren -------------------------
                    if 'Masked_Combination' in df.columns:
                        grouped = df.groupby('Masked_Combination')
                    else:
                        # ------------------------- Fallback -------------------------
                         grouped = [('All', df)]

                    for combo_name, subset in grouped:
                         
                         # ------------------------- Über einzelne Features iterieren -------------------------
                         feats_in_combo = subset['Feature'].unique()
                         num_feats = len(feats_in_combo)
                         if num_feats == 0: continue

                         # ------------------------- Kombinations-Header -------------------------
                         combo_str = str(combo_name)
                         combo_str = combo_str.replace("[", "").replace("]", "").replace("'", "").replace('"', "")
                         
                         # ------------------------- Dynamische Anpassung -------------------------
                         row_height = 4.0
                         header_height = 1.5
                         total_height = header_height + (num_feats * row_height)
                         
                         # ------------------------- Minimal A4 -------------------------
                         page_height = max(8.27, total_height)
                         
                         fig = plt.figure(figsize=(11.69, page_height))
                         
                         # ------------------------- Grid-Spec -------------------------
                         gs = fig.add_gridspec(num_feats + 1, 2, height_ratios=[0.3] + [1.0]*num_feats)
                         
                         ax_head = fig.add_subplot(gs[0, :])
                         ax_head.axis('off')
                         ax_head.text(0.5, 0.5, f"Missing Attributes: {combo_str} (Level {current_level})", 
                                      ha='center', va='center', fontsize=16, weight='bold', wrap=True)

                         # ------------------------- Einzelne Plots -------------------------
                         for idx, feature in enumerate(feats_in_combo):
                             feat_subset = subset[subset['Feature'] == feature]
                             if feat_subset.empty: continue
                             
                             # ------------------------- Metriken -------------------------
                             y_true = feat_subset['Original']
                             y_pred = feat_subset['Imputed']
                             mse = mean_squared_error(y_true, y_pred)
                             rmse = np.sqrt(mse)
                             r2 = r2_score(y_true, y_pred)

                             if current_level == 1:
                                 # ------------------------- Feature pro Kombination -------------------------
                                 rmse_per_feat[feature] = rmse
                                 all_metrics.append({'Run_ID': run_id, 'Feature': feature, 'R2': r2, 'RMSE': rmse})

                             # ------------------------- Indizes der einzelnen Ploits -------------------------
                             ax_scat = fig.add_subplot(gs[idx+1, 0])
                             ax_kde = fig.add_subplot(gs[idx+1, 1])

                             # ------------------------- Downsample um .pdf-Anzeige effizienter zu gestalten -------------------------
                             MAX_PLOT_POINTS = 10000
                             if len(feat_subset) > MAX_PLOT_POINTS:
                                 subset_plot = feat_subset.sample(n=MAX_PLOT_POINTS, random_state=42)
                                 note_sampling = f" (Sampled {MAX_PLOT_POINTS})"
                             else:
                                 subset_plot = feat_subset
                                 note_sampling = ""
                             
                             # ------------------------- Scatter-Plots -------------------------
                             sns.scatterplot(x=subset_plot['Original'], y=subset_plot['Imputed'], ax=ax_scat, alpha=0.5, hue=subset_plot['Split'], palette={'Train': 'blue', 'Test': 'red'}, legend=False)
                             low, high = min(y_true.min(), y_pred.min()) - 0.1, max(y_true.max(), y_pred.max()) + 0.1
                             ax_scat.plot([low, high], [low, high], 'k--', alpha=0.7)
                             ax_scat.set_title(f"{feature} - Correlation (R²={r2:.3f}){note_sampling}", fontsize=11)
                             ax_scat.set_xlabel("Original")
                             ax_scat.set_ylabel("Imputed")
                             
                             # ------------------------- KDE -------------------------
                             try:
                                sns.kdeplot(subset_plot['Original'], ax=ax_kde, label="Original", fill=True, alpha=0.3, color='blue', warn_singular=False)
                                sns.kdeplot(subset_plot['Imputed'], ax=ax_kde, label="Imputed", fill=True, alpha=0.3, color='orange', warn_singular=False)
                             except: pass # ------------------------- KDE fail safe -------------------------
                             ax_kde.set_title(f"{feature} - Distribution (RMSE={rmse:.3f})", fontsize=11)
                             ax_kde.legend()

                         plt.tight_layout()
                         pdf.savefig(fig)
                         plt.close(fig)
                    
                    del df
                    gc.collect()

                processed_runs[run_id] = current_files_set
                
                # ------------------------- Rename Temp to Final -------------------------
#                 if pdf_final_path.exists(): pdf_final_path.unlink()
#                 pdf_temp_path.rename(pdf_final_path)
                
                log(f"PDF saved: {pdf_path.name}")
                
                avg_rmse = np.mean(list(rmse_per_feat.values())) if rmse_per_feat else 0
                summary_stats.append({
                    "Run": run_id,
                    "Avg_RMSE": avg_rmse,
                    "Features": list(rmse_per_feat.keys())
                })
        
        except Exception as e:
            log(f"Error processing run {run_id}: {e}")
            import traceback
            traceback.print_exc()
            # ------------------------- Fehler beim Run -> Trotzdem markieren, damit Loop weitergeht -------------------------
            processed_runs[run_id] = current_files_set
            log(f"Skipping run {run_id} due to error.")

    # ------------------------- Abschluss Check -------------------------
    if done_training:
        current_run_keys = set(run_groups.keys())
        if current_run_keys.issubset(processed_runs):
             if summary_stats:
                df_summary = pd.DataFrame(summary_stats)
                csv_path = out_dir_eval / "Summary_Evaluation.csv"
                df_summary.to_csv(csv_path, index=False)
                
                sum_pdf_path = out_dir_eval / "Evaluation_Summary_Report.pdf"
                with PdfPages(sum_pdf_path) as pdf:
                    fig, ax = plt.subplots(figsize=(11.69, max(4, len(df_summary)*0.3 + 2)))
                    ax.axis('tight')
                    ax.axis('off')
                    cols = ['Run', 'Avg_RMSE']
                    ax.table(cellText=df_summary[cols].values, colLabels=cols, loc='center')
                    ax.set_title("Overall Performance Summary", fontsize=16)
                    pdf.savefig(fig)
                    plt.close()
                    
                    fig2, ax2 = plt.subplots(figsize=(11.69, 6))
                    sns.barplot(data=df_summary, x='Run', y='Avg_RMSE', ax=ax2)
                    plt.xticks(rotation=90)
                    plt.tight_layout()
                    pdf.savefig(fig2)
                    plt.close()
                    
                log(f"Global Summary PDF created: {sum_pdf_path.name}")

             log("Alle Evaluierungen abgeschlossen.")
             break
                 
    time.sleep(5)

# ------------------------- Final Report -------------------------
if all_metrics:
    df_metrics = pd.DataFrame(all_metrics)
    summary_path = out_dir_eval / "Evaluation_Summary.csv"
    df_metrics.to_csv(summary_path, index=False)
    print(f"\nZusammenfassung gespeichert: {summary_path}")
    
    pdf_path = out_dir_eval / "Evaluation_Report.pdf"
    with PdfPages(pdf_path) as pdf:
        fig, ax = plt.subplots(figsize=(11.69, 8.27))
        ax.axis('off')
        ax.text(0.5, 0.95, "VAE Imputation - Evaluation Report", ha='center', fontsize=24, weight='bold')
        ax.text(0.5, 0.90, f"Date: {time.strftime('%Y-%m-%d %H:%M')}", ha='center', fontsize=12)
        
        agg_funcs = {'R2': 'mean', 'RMSE': 'mean'}
        top_n = df_metrics.groupby('Run_ID').agg(agg_funcs).sort_values('R2', ascending=False).head(10).reset_index()
        top_n['R2'] = top_n['R2'].round(4)
        top_n['RMSE'] = top_n['RMSE'].round(4)
        
        table_data = [top_n.columns.values.tolist()] + top_n.values.tolist()
        table = ax.table(cellText=table_data, colLabels=top_n.columns, loc='center', cellLoc='center', bbox=[0.1, 0.4, 0.8, 0.4])
        pdf.savefig(fig)
        plt.close()
